# Data Cleaning

This notebook applies the cleaning steps identified during inspection
(see `data_inspection.ipynb`) and exports the results as cleaned CSV files.

In [1]:
import pandas as pd

## 1. Load Raw Data

In [2]:
INPUT_DATA_PATH = 'data/input/PART2-26-02-13_reorg'
OUTPUT_DATA_PATH = 'data/output'

train_hai = pd.read_csv(INPUT_DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(INPUT_DATA_PATH + '/train_participants.tsv', sep='\t')

## 2. Clean HAI Table

Three issues to fix:
1. **Missing `timepoint` (~1.9%) and `value` (~0.02%)**. These rows carry no
   usable measurement, so we drop them.
2. **Non-numeric `value` entries**. Coerce to numeric; anything that cannot
   convert becomes NaN and is dropped.
3. **Placeholder strain entries (`'-'`)**. Not real strains; remove them.

In [3]:
print(f'Rows before cleaning: {len(train_hai)}')

# Drop rows where timepoint or value is missing
train_hai = train_hai.dropna(subset=['timepoint', 'value'])

# Coerce value to numeric; non-convertible entries become NaN
train_hai['value'] = pd.to_numeric(train_hai['value'], errors='coerce')
train_hai = train_hai.dropna(subset=['value'])

# Remove placeholder strain entries
train_hai = train_hai[train_hai['virus_strain'] != '-']

print(f'Rows after cleaning:  {len(train_hai)}')

Rows before cleaning: 128177
Rows after cleaning:  119840


## 3. Clean Participants Table

Three issues to fix:
1. **`main_publication_author` missing ~48%** — too sparse to be useful; drop
   the column.
2. **Inconsistent `biological_sex` casing** — normalise to lowercase.
3. **Noisy `race` labels** — collapse `'race: unknown'` and `'race: other'`
   into a single `'Unknown'` category.

In [4]:
print(f'Rows before cleaning: {len(train_participants)}')

# Drop the sparse publication column
train_participants = train_participants.drop(columns=['main_publication_author'])

# Normalise biological_sex
train_participants['biological_sex'] = (
    train_participants['biological_sex'].str.lower().str.strip()
)

# Standardize race labels
train_participants['race'] = train_participants['race'].replace({
    'race: unknown': 'Unknown',
    'race: other': 'Unknown',
})

print(f'Rows after cleaning:  {len(train_participants)}')

Rows before cleaning: 3960
Rows after cleaning:  3960


## 4. Export Cleaned Data

In [5]:
train_hai.to_csv(OUTPUT_DATA_PATH + '/train_hai_cleaned.csv', index=False)
train_participants.to_csv(OUTPUT_DATA_PATH + '/train_participants_cleaned.csv', index=False)

print('Wrote train_hai_cleaned.csv         ', train_hai.shape)
print('Wrote train_participants_cleaned.csv ', train_participants.shape)

Wrote train_hai_cleaned.csv          (119840, 6)
Wrote train_participants_cleaned.csv  (3960, 16)
